# Trekking Route Safety Classification

Binary classification of mountain trekking routes as safe/dangerous from slope angle, rainfall, wolf-encounter probability, and hiker cold resistance, using scikit-learn's `DecisionTreeClassifier`. Includes a feature-engineering experiment (a combined `environmental_danger` term) evaluated against a baseline model.


## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, f1_score


## Load the dataset

In [ ]:
CSV_PATH = "../data/trekking_expedition.csv"

df = pd.read_csv(CSV_PATH)
print(df.head())
print(df.shape)


## Basic cleaning

Validates the expected schema, coerces numeric columns, drops duplicates/missing values, and enforces the valid domain for each feature (`slope_angle >= 0`, `wolf_prob` in `[0, 1]`, `rain_mm >= 0`, `is_safe` in `{0, 1}`, `cold_resistance` in `{Low, Medium, High}`).

In [ ]:
required_columns = ["slope_angle", "wolf_prob", "rain_mm", "cold_resistance", "is_safe"]
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    print(f"Dataset is missing required columns: {missing_cols}")
else:
    print("All required columns are present.")

df = df.drop_duplicates().reset_index(drop=True)

numeric_cols = ["slope_angle", "wolf_prob", "rain_mm", "is_safe"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna().reset_index(drop=True)

df = df[(df["slope_angle"] >= 0) &
        (df["wolf_prob"] >= 0) & (df["wolf_prob"] <= 1) &
        (df["rain_mm"] >= 0) &
        (df["is_safe"].isin([0, 1]))]

df = df[df["cold_resistance"].isin(["Low", "Medium", "High"])]

df["is_safe"] = df["is_safe"].astype(int)
df = df.reset_index(drop=True)

print(f"Cleaned data shape: {df.shape}")
print(f"Target distribution:\n{df['is_safe'].value_counts(normalize=True)}")


## Feature engineering — `environmental_danger`

A single combined hazard term: slope, rainfall, and wolf-encounter probability multiplied together, on the reasoning that these three risk factors compound rather than act independently.

In [ ]:
def add_environmental_danger(data):
    """Add a combined hazard feature: slope_angle * rain_mm * (1 + wolf_prob)."""
    data = data.copy()
    data["environmental_danger"] = (
        data["slope_angle"] * data["rain_mm"] * (1 + data["wolf_prob"])
    )
    return data


## Preprocessing

One-hot encodes `cold_resistance` and standard-scales the numeric columns, fitting both transformers on the training split only to avoid validation-set leakage (see the project README for the full discussion).

In [ ]:
def preprocess_train_validation(X_train, X_val):
    """Fit encoder/scaler on X_train only, then transform both splits."""
    X_train = X_train.copy()
    X_val = X_val.copy()

    categorical_columns = ["cold_resistance"]
    numeric_columns = [col for col in X_train.columns if col not in categorical_columns]

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder.fit(X_train[categorical_columns])

    train_cat_encoded = encoder.transform(X_train[categorical_columns])
    val_cat_encoded = encoder.transform(X_val[categorical_columns])

    cat_feature_names = encoder.get_feature_names_out(categorical_columns)
    train_cat_df = pd.DataFrame(train_cat_encoded, columns=cat_feature_names, index=X_train.index)
    val_cat_df = pd.DataFrame(val_cat_encoded, columns=cat_feature_names, index=X_val.index)

    scaler = StandardScaler()
    scaler.fit(X_train[numeric_columns])

    train_num_scaled = scaler.transform(X_train[numeric_columns])
    val_num_scaled = scaler.transform(X_val[numeric_columns])

    train_num_df = pd.DataFrame(train_num_scaled, columns=numeric_columns, index=X_train.index)
    val_num_df = pd.DataFrame(val_num_scaled, columns=numeric_columns, index=X_val.index)

    X_train_processed = pd.concat([train_num_df, train_cat_df], axis=1)
    X_val_processed = pd.concat([val_num_df, val_cat_df], axis=1)
    return X_train_processed, X_val_processed


## Train and evaluate

In [ ]:
def train_and_evaluate(data, experiment_name):
    """Split, preprocess, fit a DecisionTreeClassifier, and report metrics."""
    print("\n" + "=" * 70)
    print(experiment_name)
    print("=" * 70)

    X = data.drop(columns=["is_safe"])
    y = data["is_safe"]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    X_train_processed, X_val_processed = preprocess_train_validation(X_train, X_val)

    model = DecisionTreeClassifier(
        max_depth=4,
        min_samples_split=25,
        min_samples_leaf=12,
        random_state=42,
    )
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_val_processed)

    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1-score : {f1:.4f}")

    return {
        "experiment": experiment_name,
        "accuracy": accuracy,
        "f1_score": f1,
        "model": model,
        "X_train_processed": X_train_processed,
    }


## Run experiments: baseline vs. engineered feature

In [ ]:
df_baseline = df.copy()
result_baseline = train_and_evaluate(
    df_baseline, "Baseline Decision Tree - Without environmental_danger"
)

df_engineered = add_environmental_danger(df)
result_engineered = train_and_evaluate(
    df_engineered, "Decision Tree - With environmental_danger"
)


## Final comparison

In [ ]:
comparison = pd.DataFrame([
    {
        "experiment": result_baseline["experiment"],
        "accuracy": result_baseline["accuracy"],
        "f1_score": result_baseline["f1_score"],
    },
    {
        "experiment": result_engineered["experiment"],
        "accuracy": result_engineered["accuracy"],
        "f1_score": result_engineered["f1_score"],
    },
])
print(comparison.round(4).to_string(index=False))

acc_change = result_engineered["accuracy"] - result_baseline["accuracy"]
f1_change = result_engineered["f1_score"] - result_baseline["f1_score"]
print(f"\nChange in Accuracy : {acc_change:+.4f}")
print(f"Change in F1-score : {f1_change:+.4f}")


In [ ]:
# Visualize the trained tree (engineered model) to support the decision-rule
# discussion in the project README.
plt.figure(figsize=(18, 8))
plot_tree(
    result_engineered["model"],
    feature_names=result_engineered["X_train_processed"].columns,
    class_names=["dangerous", "safe"],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Decision Tree — With environmental_danger feature")
plt.tight_layout()
plt.show()
